# Arabic OCR Evaluation

Runs the six candidate models plus Gemini on `test_images/`, scores them against the ground truth, and writes `results/results.json`.
Method: [docs/evaluation_strategy.md](../docs/evaluation_strategy.md).

**Runtime:** GPU. An **L4 or A100** (Colab Pro) is best: they support bf16, which the Gemma-based legal model needs. A free **T4** also works: the code switches to fp16, and loads the legal model in 4-bit with fp32 compute.

Each model runs in its own process, so a crash or out-of-memory error in one model never stops the others, and GPU memory is fully freed between models.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code and test images

In [ ]:
!git clone https://github.com/sir-thabet-work/OCR-Benchmark-.git ocr-benchmark
%cd ocr-benchmark

## 2. Install

In [ ]:
!pip install -q -U -r eval/requirements.txt

## 3. Keys

Add these in Colab **Secrets** (key icon on the left):
- `GEMINI_API_KEY`: needed for the Gemini run.
- `HF_TOKEN`: optional; all six models are public, but a token avoids download rate limits.

Set `GEMINI_MODEL` to the newest Gemini model id.

In [ ]:
import os
from google.colab import userdata

for key in ('GEMINI_API_KEY', 'HF_TOKEN'):
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        print(f'{key} not set')

os.environ['GEMINI_MODEL'] = 'gemini-2.5-pro'  # replace with the newest Gemini model id

## 4. Run the models

Outputs go to `outputs/<model>/`. Re-running a model overwrites its folder, so you can run one again on its own after fixing a problem.

In [ ]:
MODELS = ['qari', 'baseer', 'legal', 'katib', 'waqf', 'sherif', 'gemini']

for m in MODELS:
    !python eval/run_models.py {m}

**If a model failed to load:** its `outputs/<model>/_run.json` contains `load_error`.
The most likely case is `waqf`, whose custom PaddleOCR-VL code may not match the newest transformers. Retry it with the version range the cards were written for, then restore the latest:

```
!pip install -q transformers==4.57.6
!python eval/run_models.py waqf
!pip install -q -U transformers
```

## 5. Score

Fill in prices to get cost per 1,000 pages: the hourly price of this GPU, and the Gemini per-1M-token prices from the Gemini pricing page.

In [ ]:
GPU_HOURLY_USD = None           # e.g. 0.80
GEMINI_INPUT_USD_PER_M = None   # e.g. 1.25
GEMINI_OUTPUT_USD_PER_M = None  # e.g. 10.0

price_args = ''
if GPU_HOURLY_USD is not None:
    price_args += f' --gpu-hourly-usd {GPU_HOURLY_USD}'
if GEMINI_INPUT_USD_PER_M is not None and GEMINI_OUTPUT_USD_PER_M is not None:
    price_args += f' --gemini-input-usd-per-m {GEMINI_INPUT_USD_PER_M} --gemini-output-usd-per-m {GEMINI_OUTPUT_USD_PER_M}'

!python eval/score.py {price_args}

## 6. Look at the results

In [ ]:
import json
import pandas as pd

res = json.load(open('results/results.json', encoding='utf-8'))

leaderboard = pd.DataFrame({m: d['summary'] for m, d in res['models'].items()}).T
cols = ['cer_mean', 'cer_median', 'cer_no_diacritics_mean', 'wer_mean', 'cer_p95', 'failure_rate',
        'latency_median_s', 'peak_vram_gb', 'cost_per_1k_pages_usd']
display(leaderboard[cols].sort_values('cer_median'))

# CER matrix: models x images
matrix = pd.DataFrame({m: {img: v['cer'] for img, v in d['images'].items()} for m, d in res['models'].items()})
display(matrix.style.background_gradient(cmap='RdYlGn_r', vmin=0, vmax=1).format('{:.3f}'))

## 7. Save the results

Download the outputs and `results.json`, then commit them to the repo, so the manual review (table structure, reading order) and the lab page can use them.

In [ ]:
!zip -qr ocr_results.zip outputs results
from google.colab import files
files.download('ocr_results.zip')